<a href="https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/NB_proteome_library_builder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proteome Library Builder

**Producer notebook.** This is the *only* notebook that writes the shared proteome library. It
assembles the taxon table (your curated core + GTDB phylogenetic sampling), fetches one proteome
per organism, and builds one local BLAST database per organism. The search notebook
(`NB03_…`) only *reads* what this produces.

Pipeline: curated table → GTDB enrichment → GTDB sampling (fill thin phyla) → **final table** →
fetch proteomes → build BLAST databases → library status.

Everything lands in `Databases/proteome_library/` (`proteomes/`, `blastdbs/`, `manifest/`) and is
cached, so re-running only does the new work. Run the cells in order.

## 0. Setup — mount Drive and create the library folders

In [1]:
import subprocess, sys, os
# dependencies: Biopython, NCBI 'datasets' CLI, BLAST+ (idempotent, safe to re-run)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython==1.85", "pandas"], check=True)
if not os.path.exists("/usr/local/bin/datasets"):
    subprocess.run("curl -sSL 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets'"
                   " -o /usr/local/bin/datasets && chmod +x /usr/local/bin/datasets", shell=True, check=True)
if subprocess.run("which blastp", shell=True, capture_output=True).returncode != 0:
    subprocess.run("apt-get -qq update && apt-get -qq install -y ncbi-blast+", shell=True, check=True)
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path

PROJECT_DRIVE = "/content/drive/MyDrive/BIOINFO4-5203-F26"   # your project folder on Drive

PROJECT      = Path(PROJECT_DRIVE)
DB_ROOT      = PROJECT / "Databases" / "proteome_library"    # the shared library this notebook writes
PROTEOME_DIR = DB_ROOT / "proteomes"                          # one <taxid>.faa per organism
DB_DIR       = DB_ROOT / "blastdbs"                           # one BLAST db per organism
MANIFEST_DIR = DB_ROOT / "manifest"
GTDB_DIR     = PROJECT / "Databases" / "gtdb"                 # cached GTDB metadata
for d in (PROTEOME_DIR, DB_DIR, MANIFEST_DIR, GTDB_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("library ->", DB_ROOT)


Mounted at /content/drive
library -> /content/drive/MyDrive/BIOINFO4-5203-F26/Databases/proteome_library


## 1. Configuration

Three groups of settings: which **inputs** to read, how to **sample** GTDB to broaden coverage, and
how to **fetch** proteomes.

In [2]:
from Bio import Entrez

# ---------------- INPUTS / OUTPUTS ----------------
CURATED_CSV       = "Taxon_library_v1.csv"        # your hand-curated core table, in Databases/proteome_library/
ENRICHED_CSV      = "Taxon_library_v1_gtdb.csv"   # intermediate: curated + GTDB columns
FINAL_LIBRARY_CSV = "Taxon_library_final.csv"     # the production table this notebook builds the library from
Entrez.email      = "your.name@your.institution.edu"   # NCBI requires a contact email
NCBI_API_KEY      = ""
Entrez.api_key    = NCBI_API_KEY or None
Entrez.tool       = "proteome_library_builder"

# ---------------- GTDB SOURCE ----------------
GTDB_RELEASE = "232"                              # current GTDB release; see gtdb.ecogenomic.org/downloads
_base = f"https://data.gtdb.ecogenomic.org/releases/release{GTDB_RELEASE}/{GTDB_RELEASE}.0"
GTDB_BAC_URL = f"{_base}/bac120_metadata_r{GTDB_RELEASE}.tsv.gz"
GTDB_AR_URL  = f"{_base}/ar53_metadata_r{GTDB_RELEASE}.tsv.gz"

# ---------------- SAMPLING (broaden coverage) ----------------
TARGET_RICH   = 3     # representatives per bioenergetically rich phylum
TARGET_OTHER  = 1     # representatives per other phylum (tree-of-life backbone)
SAMPLE_MIN_COMPLETENESS  = 90.0   # CheckM completeness floor
SAMPLE_MAX_CONTAMINATION = 5.0    # CheckM contamination ceiling
PREFER_ISOLATES    = True         # take isolates before MAGs/SAGs
ALLOW_MAG_FALLBACK = False         # skip a phylum with no qualifying isolate (keeps 'no_hits' meaningful)
RICH_PHYLA = {
    "pseudomonadota", "proteobacteria", "actinomycetota", "actinobacteriota",
    "bacillota", "firmicutes", "bacteroidota", "campylobacterota", "nitrospirota",
    "aquificota", "deinococcota", "chloroflexota", "desulfobacterota",
    "thermodesulfobacteriota", "halobacteriota", "methanobacteriota", "thermoproteota",
}
# To grow toward ~250 taxa, raise TARGET_OTHER to 2 (or TARGET_RICH to 4); the report at the
# end of Section 6 prints the running count so you can tune it.

# ---------------- PROTEOME FETCH ----------------
ASSEMBLY_SOURCE  = "RefSeq"        # preferred proteome source; falls back to GenBank if needed
ALLOW_NONREF_FETCH = True          # allow the strain's latest annotated assembly if no reference genome
FORCE_REFETCH    = False           # re-download proteomes even if cached
FORCE_REBUILD_DB = False           # rebuild BLAST databases even if present
MAX_RETRIES  = 3
BACKOFF_START = 5.0


## 2. Load the curated core table

In [3]:
import pandas as pd
cur_path = DB_ROOT / CURATED_CSV
if not cur_path.exists():
    raise FileNotFoundError(f"Curated table not found at {cur_path}")
curated = pd.read_csv(cur_path)

def pick_taxid_col(df):
    for c in ("resolved_taxid", "NCBI_TaxID", "taxid", "species_taxid"):
        if c in df.columns:
            return c
    raise ValueError(f"No TaxID column in {list(df.columns)}")
TAXCOL = pick_taxid_col(curated)
print(f"curated core: {len(curated)} taxa (TaxID column '{TAXCOL}')")


curated core: 149 taxa (TaxID column 'NCBI_TaxID')


## 3. Download GTDB metadata (cached)

GTDB (Genome Taxonomy Database, University of Queensland) provides a genome-based, rank-normalized
taxonomy and a CheckM quality score for every genome. A few hundred MB, downloaded once.

In [4]:
def fetch_url(url, dest):
    if dest.exists() and dest.stat().st_size > 0:
        print("cached:", dest.name); return
    print("downloading", dest.name, "...")
    rc = subprocess.run(["wget", "-q", "-O", str(dest), url]).returncode
    if rc != 0 or not dest.exists() or dest.stat().st_size == 0:
        if dest.exists(): dest.unlink()
        raise RuntimeError(f"download failed: {url}\nGet the current path at https://gtdb.ecogenomic.org/downloads")
    print("  ->", dest.stat().st_size // (1024*1024), "MB")

bac_path = GTDB_DIR / f"bac120_metadata_r{GTDB_RELEASE}.tsv.gz"
ar_path  = GTDB_DIR / f"ar53_metadata_r{GTDB_RELEASE}.tsv.gz"
fetch_url(GTDB_BAC_URL, bac_path)
fetch_url(GTDB_AR_URL,  ar_path)


cached: bac120_metadata_r232.tsv.gz
cached: ar53_metadata_r232.tsv.gz


## 4. Load GTDB metadata and index it

Loads only the columns we need (so the large files fit in memory), detecting names that shift
between releases. Builds two lookups: species representatives, and TaxID → best genome.

In [5]:
def load_meta(path):
    hdr = pd.read_csv(path, sep="\t", nrows=0, compression="gzip").columns.tolist()
    def pick(names, contains=None):
        for n in names:
            if n in hdr: return n
        if contains:
            for c in hdr:
                if contains in c.lower(): return c
        return None
    cols = {
        "acc":     pick(["accession"]),
        "taxid":   pick(["ncbi_taxid"]),
        "gtdbtax": pick(["gtdb_taxonomy"]),
        "rep":     pick(["gtdb_representative"]),
        "comp":    pick(["checkm2_completeness", "checkm_completeness"], contains="completeness"),
        "cont":    pick(["checkm2_contamination", "checkm_contamination"], contains="contamination"),
        "name":    pick(["ncbi_organism_name"]),
        "gcat":    pick(["ncbi_genome_category"], contains="genome_category"),
    }
    use = [c for c in cols.values() if c]
    df = pd.read_csv(path, sep="\t", usecols=use, compression="gzip", dtype=str)
    return df.rename(columns={v: k for k, v in cols.items() if v})

meta = pd.concat([load_meta(bac_path), load_meta(ar_path)], ignore_index=True)
meta["taxid_i"] = pd.to_numeric(meta.get("taxid"), errors="coerce")
meta = meta.dropna(subset=["taxid_i"]).copy(); meta["taxid_i"] = meta["taxid_i"].astype(int)
meta["isrep"]   = meta.get("rep", "").astype(str).str.lower().isin(["t", "true"])
meta["species"] = meta["gtdbtax"].str.extract(r"(s__[^;]*)")

reps     = meta[meta["isrep"]].drop_duplicates("species").set_index("species")
by_taxid = meta.sort_values("isrep", ascending=False).drop_duplicates("taxid_i").set_index("taxid_i")
def _norm(s): return "".join(ch.lower() for ch in str(s) if ch.isalnum())
if "name" in meta.columns:
    meta["name_norm"] = meta["name"].map(_norm)
    by_name = meta.sort_values("isrep", ascending=False).drop_duplicates("name_norm").set_index("name_norm")
else:
    by_name = None
print(f"GTDB: {len(meta):,} genomes, {len(reps):,} species representatives")


GTDB: 901,341 genomes, 199,923 species representatives


## 5. Enrich the curated table with GTDB

Attaches each curated prokaryote's GTDB lineage, CheckM quality, and representative accession.
Eukaryotes are not in GTDB and pass through unmatched (expected).

In [6]:
import re
def _strip_prefix(a):
    a = str(a); return a[3:] if a[:3] in ("RS_", "GB_") else a   # RS_GCF_... -> GCF_...
def _tax(s, rank):
    m = re.search(rf"{rank}__([^;]*)", str(s)); return m.group(1).strip() if m else ""
def _rep_for(row):
    sp = row.get("species")
    return (reps.loc[sp].to_dict() if (sp and sp in reps.index) else row.to_dict())

records = []
for _, lr in curated.iterrows():
    out = {"gtdb_taxonomy": "", "accession": "", "checkm_completeness": "",
           "checkm_contamination": "", "gtdb_match": "none"}
    try:    tx = int(float(lr[TAXCOL]))
    except (TypeError, ValueError): tx = None
    row, how = None, "none"
    if tx is not None and tx in by_taxid.index:
        row, how = by_taxid.loc[tx], "taxid"
    elif by_name is not None:
        nm = _norm(lr.get("Organism", lr.get("organism", "")))
        if nm and nm in by_name.index:
            row, how = by_name.loc[nm], "name"
    if row is not None:
        rep = _rep_for(row)
        out.update(gtdb_taxonomy=rep.get("gtdbtax", ""), accession=_strip_prefix(rep.get("acc", "")),
                   checkm_completeness=rep.get("comp", ""), checkm_contamination=rep.get("cont", ""),
                   gtdb_match=how)
    records.append(out)

enr = pd.concat([curated.reset_index(drop=True), pd.DataFrame(records)], axis=1)
enr.to_csv(DB_ROOT / ENRICHED_CSV, index=False)
matched = (enr["gtdb_match"] != "none").sum()
print(f"GTDB-matched {matched}/{len(enr)} curated taxa (unmatched are mostly eukaryotes).")


GTDB-matched 121/149 curated taxa (unmatched are mostly eukaryotes).


## 6. Sample GTDB representatives to broaden coverage

Adds isolate representatives to fill thin phyla — one per phylum, three in the rich phyla — spread
across classes, quality-gated, and **additive** (only tops up toward the target; never touches your
curated taxa). Writes the production table `FINAL_LIBRARY_CSV`.

In [7]:
pool = meta[meta["isrep"]].copy()
pool["phylum"]  = pool["gtdbtax"].map(lambda s: _tax(s, "p"))
pool["klass"]   = pool["gtdbtax"].map(lambda s: _tax(s, "c"))
pool["domain"]  = pool["gtdbtax"].map(lambda s: _tax(s, "d"))
pool["species"] = pool["gtdbtax"].map(lambda s: _tax(s, "s"))
pool["comp_f"]  = pd.to_numeric(pool.get("comp"), errors="coerce")
pool["cont_f"]  = pd.to_numeric(pool.get("cont"), errors="coerce")
if "gcat" in pool.columns:
    _gc = pool["gcat"].astype(str).str.lower()
    pool["is_isolate"] = ~_gc.str.contains("metagenome|single cell|environmental", na=False)
else:
    print("NOTE: no genome-category column; treating all as isolates."); pool["is_isolate"] = True
pool = pool[(pool["comp_f"] >= SAMPLE_MIN_COMPLETENESS) &
            (pool["cont_f"] <= SAMPLE_MAX_CONTAMINATION) & (pool["phylum"] != "")].copy()

enr_phylum = enr["gtdb_taxonomy"].map(lambda s: _tax(s, "p"))
have = enr_phylum[enr_phylum != ""].value_counts().to_dict()
existing_species = set(enr["gtdb_taxonomy"].map(lambda s: _tax(s, "s")))
existing_acc = set(enr["accession"].astype(str))

def target_for(ph): return TARGET_RICH if ph.lower() in RICH_PHYLA else TARGET_OTHER
def pick_from_phylum(cands, need):
    iso = cands[cands["is_isolate"]]
    if PREFER_ISOLATES:
        base = iso
        if len(base) < need and ALLOW_MAG_FALLBACK:
            base = pd.concat([iso, cands[~cands.index.isin(iso.index)]])
    else:
        base = cands
    if len(base) == 0: return []
    base = base.sort_values(["is_isolate", "comp_f"], ascending=[False, False])
    picked, used = [], set()
    for _, r in base.iterrows():                 # one per class first (phylogenetic spread)
        if len(picked) >= need: break
        if r["klass"] in used: continue
        picked.append(r); used.add(r["klass"])
    for _, r in base.iterrows():                 # then fill ignoring class
        if len(picked) >= need: break
        if any(r["acc"] == p["acc"] for p in picked): continue
        picked.append(r)
    return picked

new_rows, added_by_phylum = [], {}
for phylum, grp in pool.groupby("phylum"):
    need = max(0, target_for(phylum) - have.get(phylum, 0))
    if need == 0: continue
    grp = grp[~grp["species"].isin(existing_species) & ~grp["acc"].map(_strip_prefix).isin(existing_acc)]
    if len(grp) == 0: continue
    for r in pick_from_phylum(grp, need):
        acc = _strip_prefix(r["acc"])
        if acc in existing_acc: continue
        existing_acc.add(acc)
        new_rows.append({
            "Tier": "8 - GTDB backbone", "Category": "GTDB phylum coverage",
            "Organism": r.get("name", "") or r["species"].replace("s__", ""),
            "Domain": r["domain"], "Clade_or_Phylum": r["phylum"],
            "NCBI_TaxID": r.get("taxid", ""), "TaxID_status": "gtdb_rep", "CI_form": "",
            "Rationale": f"GTDB isolate representative ({r['phylum']}) coverage top-up",
            "Source": "gtdb_sampled", "gtdb_taxonomy": r["gtdbtax"], "accession": acc,
            "checkm_completeness": r.get("comp", ""), "checkm_contamination": r.get("cont", ""),
            "gtdb_match": "gtdb_rep"})
        added_by_phylum[phylum] = added_by_phylum.get(phylum, 0) + 1

new_df = pd.DataFrame(new_rows)
for c in enr.columns:
    if c not in new_df.columns: new_df[c] = ""
if len(new_df): new_df = new_df[enr.columns]
final = pd.concat([enr, new_df], ignore_index=True)
final.to_csv(DB_ROOT / FINAL_LIBRARY_CSV, index=False)

print(f"Added {len(new_rows)} taxa across {len(added_by_phylum)} phyla; "
      f"final library = {len(final)} taxa.\nWritten: {DB_ROOT / FINAL_LIBRARY_CSV}\n")
after = dict(have)
for ph, n in added_by_phylum.items(): after[ph] = after.get(ph, 0) + n
print(f"{'phylum':30s} before after")
for ph in sorted(set(list(have) + list(added_by_phylum)), key=lambda p: -after.get(p, 0)):
    grew = "  +" + str(added_by_phylum[ph]) if ph in added_by_phylum else ""
    print(f"{ph[:30]:30s} {have.get(ph,0):6d} {after.get(ph,0):5d}{grew}")


Added 60 taxa across 54 phyla; final library = 209 taxa.
Written: /content/drive/MyDrive/BIOINFO4-5203-F26/Databases/proteome_library/Taxon_library_final.csv

phylum                         before after
Cyanobacteriota                    82    82
Pseudomonadota                     13    13
Chloroflexota                       5     5
Bacillota                           1     3  +2
Thermoproteota                      2     3  +1
Methanobacteriota                   0     3  +3
Aquificota                          1     3  +2
Bacteroidota                        2     3  +1
Desulfobacterota                    1     3  +2
Actinomycetota                      1     3  +2
Deinococcota                        2     3  +1
Campylobacterota                    2     3  +1
Halobacteriota                      2     3  +1
Nitrospirota                        2     3  +1
Verrucomicrobiota                   2     2
Thermoplasmatota                    0     1  +1
UBA10199                            0     1  

## 7. Proteome + database helpers

Fetch one organism's proteome and build its BLAST database. GTDB-sampled taxa are fetched **by
accession** (the exact representative assembly); curated taxa are fetched by TaxID with a
reference → latest-annotated → GenBank fallback.

In [8]:
import time, json, shutil, tempfile, zipfile, glob, random
from datetime import datetime, timezone
EVENTS = MANIFEST_DIR / "build_events.jsonl"

def utc_now(): return datetime.now(timezone.utc).isoformat()
def atomic_text(path, text):
    tmp = path.with_name(path.name + ".tmp"); tmp.write_text(text, encoding="utf-8"); tmp.replace(path)
def save_json(path, data): atomic_text(path, json.dumps(data, indent=2, ensure_ascii=False) + "\n")
def event(kind, **d):
    with EVENTS.open("a", encoding="utf-8") as h: h.write(json.dumps({"time": utc_now(), "kind": kind, **d}) + "\n")
def run_cmd(cmd, timeout=1800):
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout); return p.returncode, p.stdout, p.stderr
def retry_call(fn, *, what):
    for attempt in range(MAX_RETRIES + 1):
        try: return fn()
        except Exception as exc:
            if attempt == MAX_RETRIES: raise
            delay = min(120, BACKOFF_START * 2 ** attempt) + random.uniform(0, 1)
            print(f"  {what}: retry {attempt+1}/{MAX_RETRIES} in {delay:.0f}s"); time.sleep(delay)

def proteome_path(taxid): return PROTEOME_DIR / f"{taxid}.faa"

def _extract_best_faa(zpath, dest):
    exdir = zpath.parent / "x"; shutil.rmtree(exdir, ignore_errors=True)
    with zipfile.ZipFile(zpath) as z: z.extractall(exdir)
    faas = glob.glob(str(exdir / "**" / "protein.faa"), recursive=True)
    if not faas: return False
    shutil.copy(max(faas, key=lambda f: os.path.getsize(f)), dest)   # most complete assembly
    return dest.stat().st_size > 0

def _download_by_accession(acc, dest):
    tmp = Path(tempfile.mkdtemp()); zp = tmp / "p.zip"
    try:
        rc, _, _ = run_cmd(["datasets", "download", "genome", "accession", str(acc),
                            "--include", "protein", "--filename", str(zp)])
        return zp.exists() and _extract_best_faa(zp, dest) if rc == 0 else False
    finally: shutil.rmtree(tmp, ignore_errors=True)

def _download_by_taxid(taxid, species_taxid, dest):
    tmp = Path(tempfile.mkdtemp()); zp = tmp / "p.zip"
    plans = [("RefSeq", str(taxid), ["--reference"])]
    if species_taxid and species_taxid != taxid:
        plans.append(("RefSeq", str(species_taxid), ["--reference"]))
    if ALLOW_NONREF_FETCH:
        plans.append(("RefSeq",  str(taxid), ["--annotated", "--assembly-version", "latest"]))
        plans.append(("GenBank", str(taxid), ["--annotated", "--assembly-version", "latest"]))
    try:
        for source, tid, extra in plans:
            if zp.exists(): zp.unlink()
            rc, _, _ = run_cmd(["datasets", "download", "genome", "taxon", tid, "--include", "protein",
                                "--assembly-source", source, "--filename", str(zp)] + extra)
            if rc == 0 and zp.exists() and _extract_best_faa(zp, dest): return True
        return False
    finally: shutil.rmtree(tmp, ignore_errors=True)

def ensure_proteome(t):
    """proteomes/<taxid>.faa: use an existing/override file, else fetch by accession, else by TaxID."""
    taxid = t["taxid"]; p = proteome_path(taxid)
    if t.get("proteome_file"):
        src = Path(t["proteome_file"]);  src = src if src.is_absolute() else DB_ROOT / t["proteome_file"]
        if not src.exists(): raise FileNotFoundError(f"proteome_file not found: {src}")
        if p.resolve() != src.resolve(): shutil.copy(src, p)
        return "override"
    if p.exists() and p.stat().st_size > 0 and not FORCE_REFETCH:
        return "cached"
    if t.get("accession"):                                   # GTDB representative: exact assembly
        if retry_call(lambda: _download_by_accession(t["accession"], p), what=f"fetch acc {t['accession']}"):
            return "fetched(acc)"
    if retry_call(lambda: _download_by_taxid(taxid, t.get("species_taxid"), p), what=f"fetch {taxid}"):
        return "fetched(taxid)"
    raise RuntimeError("no annotated proteome (supply a proteome_file or a same-tier substitute)")

def db_prefix(taxid): return DB_DIR / str(taxid)
def db_ready(taxid):  return bool(glob.glob(str(db_prefix(taxid)) + ".p*"))
def build_db(taxid):
    if db_ready(taxid) and not FORCE_REBUILD_DB: return "cached"
    for f in glob.glob(str(db_prefix(taxid)) + ".p*"): os.remove(f)
    rc, _, err = run_cmd(["makeblastdb", "-in", str(proteome_path(taxid)), "-dbtype", "prot", "-out", str(db_prefix(taxid))])
    if rc != 0: raise RuntimeError(f"makeblastdb failed for {taxid}: {err[:300]}")
    return "built"


## 8. Build the library — fetch proteomes and build databases

For every taxon in the final table: fetch its proteome and build its BLAST database (both cached).
Failures are recorded and skipped so one missing genome never blocks the rest. This is the slow,
one-time step; re-running only handles new or changed taxa.

In [9]:
final = pd.read_csv(DB_ROOT / FINAL_LIBRARY_CSV)
FTAX = pick_taxid_col(final)

def _as_taxid(v):
    try: return int(float(v))
    except (TypeError, ValueError): return None

targets = []
for _, r in final.iterrows():
    tx = _as_taxid(r.get(FTAX))
    if tx is None: continue                          # need a TaxID to key the proteome/db file
    targets.append({"taxid": tx, "label": str(r.get("Organism", f"taxid_{tx}")).strip(),
                    "species_taxid": _as_taxid(r.get("species_taxid")),
                    "accession": (str(r["accession"]).strip() if "accession" in final.columns
                                  and pd.notna(r.get("accession")) and str(r.get("accession")).strip() else ""),
                    "proteome_file": (str(r["proteome_file"]).strip() if "proteome_file" in final.columns
                                      and pd.notna(r.get("proteome_file")) else "")})

status_rows = []
for i, t in enumerate(targets, 1):
    row = {"taxid": t["taxid"], "label": t["label"], "proteome": "", "database": "", "n_proteins": "", "error": ""}
    print(f"[{i}/{len(targets)}] {t['label']} (TaxID {t['taxid']})")
    try:
        row["proteome"] = ensure_proteome(t)
        row["database"] = build_db(t["taxid"])
        row["n_proteins"] = sum(1 for ln in proteome_path(t["taxid"]).open() if ln.startswith(">"))
        event("library_ready", taxid=t["taxid"], **{k: row[k] for k in ("proteome", "database", "n_proteins")})
        print(f"    proteome={row['proteome']}, db={row['database']}, proteins={row['n_proteins']}")
    except Exception as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"
        event("library_error", taxid=t["taxid"], error=row["error"]); print(f"    ERROR: {row['error']}")
    status_rows.append(row)

LIB_STATUS_CSV = MANIFEST_DIR / "library_status.csv"
pd.DataFrame(status_rows).to_csv(LIB_STATUS_CSV, index=False)
ok = sum(1 for r in status_rows if r["database"] and not r["error"])
print(f"\nLibrary built: {ok}/{len(targets)} taxa ready. Status -> {LIB_STATUS_CSV}")


[1/197] Aquifex aeolicus VF5 (TaxID 224324)
    proteome=cached, db=cached, proteins=1738
[2/197] Paracoccus denitrificans (TaxID 266)
    proteome=cached, db=cached, proteins=5014
[3/197] Escherichia coli K-12 subMG1655 (TaxID 83333)
    proteome=cached, db=cached, proteins=4300
[4/197] Thermus thermophilus HB8 (TaxID 300852)
    proteome=cached, db=cached, proteins=2150
[5/197] Thermus thermophilus JL-18 (TaxID 798128)
    proteome=cached, db=cached, proteins=2394
[6/197] Rhodobacter capsulatus SB1003 (TaxID 272942)
    proteome=cached, db=cached, proteins=3607
[7/197] Rickettsia prowazekii (TaxID 782)
    proteome=cached, db=cached, proteins=831
[8/197] Yarrowia lipolytica (TaxID 4952)
    proteome=cached, db=cached, proteins=7918
[9/197] Tetrahymena thermophila (TaxID 312017)
    proteome=cached, db=cached, proteins=26996
[10/197] Homo sapiens (TaxID 9606)
    proteome=cached, db=cached, proteins=136807
[11/197] Bos taurus (TaxID 9913)
    proteome=cached, db=cached, proteins=64745

## 9. Library status

In [10]:
st = pd.read_csv(MANIFEST_DIR / "library_status.csv")
ready  = st[st["database"].notna() & (st["database"].astype(str).str.strip() != "")]
failed = st[st["error"].notna() & (st["error"].astype(str).str.strip() != "")]   # NaN = success, not a failure
print(f"Ready: {len(ready)} / {len(st)}")
print("Proteome source:", st["proteome"].value_counts().to_dict())
if len(failed):
    print(f"\nGenuinely failed ({len(failed)}) - supply a proteome_file or a same-tier substitute:")
    for _, r in failed.iterrows():
        print(f"  - {r['label']} (TaxID {r['taxid']}): {r['error']}")
else:
    print("\nNo failures.")
print("\nThe search notebook reads this finished library; it does not rebuild it.")


Ready: 194 / 197
Proteome source: {'cached': 133, 'fetched(acc)': 60, 'fetched(taxid)': 1}

Genuinely failed (3) - supply a proteome_file or a same-tier substitute:
  - Chroococcidiopsis sp. PCC 6712 (TaxID 118675): RuntimeError: no annotated proteome (supply a proteome_file or a same-tier substitute)
  - Cyanothece sp. PCC 7425 (TaxID 395961): RuntimeError: no annotated proteome (supply a proteome_file or a same-tier substitute)
  - Fischerella sp. PCC 9431 (TaxID 1173023): RuntimeError: no annotated proteome (supply a proteome_file or a same-tier substitute)

The search notebook reads this finished library; it does not rebuild it.
